In [0]:
%sql

USE CATALOG gizmobox;
USE SCHEMA landing;

select * 
from read_files('/Volumes/gizmobox/landing/operational_data/addresses',
    format => 'csv',
    delimiter => '\t',
    header => true)

In [0]:
%sql

USE CATALOG gizmobox;
USE SCHEMA bronze;

create view if not exists v_addresses 
AS 
select * FROM read_files('/Volumes/gizmobox/landing/operational_data/addresses',
    format => 'csv',
    delimiter => '\t',
    header => true)

In [0]:
%sql
select * from v_addresses;

#### Move data from Workspace to volume

In [0]:
# Copy data from workspace to dbfs
dbutils.fs.cp(
"/Workspace/Users/nhuynh27@gmu.edu/Databricks/Gizmobox_project/data/payments",
"/Volumes/gizmobox/landing/operational_data/payments",
recurse=True
)

#### Adding headers to raw payment data using PySpark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder \
    .appName("payments") \
    .getOrCreate()

In [0]:
schema = StructType([
    StructField("payment_id", IntegerType(), True),
    StructField("order_id", IntegerType(), True),
    StructField("payment_timestamp", TimestampType(), True),
    StructField("payment_status", IntegerType(), True),
    StructField("payment_method", StringType(), True)
])

In [0]:
df = spark.read.csv("/Volumes/gizmobox/landing/operational_data/payments", header=False, schema=schema)

In [0]:
df.createOrReplaceTempView("payment_temp")

##### Register temp view to unity catalog

In [0]:
%sql
USE CATALOG gizmobox;
USE SCHEMA bronze;


CREATE VIEW IF NOT EXISTS v_payments AS
SELECT *
FROM csv.`/Volumes/gizmobox/landing/operational_data/payments`;